In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
def llm(prompt):
    # Use chat.completions instead of response
    response = openai_client.chat.completions.create(
        model='gpt-5.4-mini',  # Ensure your model name is accurate
        messages=[{"role": "user", "content": prompt}]
    )
    # Extract text from choices
    return response.choices[0].message.content


In [4]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

Absolutely — in most cases you can still join after the course has started.

A few things depend on the course:
- **How far along it is**: if it’s early, you may be able to catch up easily.
- **Whether recordings/materials are available**: some courses let late joiners review missed sessions.
- **Enrollment rules**: some programs have deadlines or limited seats.

If you want, I can help you draft a quick message to the instructor or course coordinator asking to join.


In [5]:
llm("What is the meaning of life?")

'There isn’t one universally agreed-upon answer.\n\nDifferent people and traditions see the meaning of life as:\n- **Connection**: loving and being loved, building community\n- **Growth**: learning, improving, becoming wiser\n- **Purpose**: serving others, creating, contributing\n- **Experience**: appreciating beauty, joy, curiosity, and the present moment\n- **Faith or spirituality**: fulfilling a divine or cosmic purpose\n\nA practical answer is: **the meaning of life is something you create through the way you live**.\n\nIf you want, I can also give you:\n1. a philosophical answer,  \n2. a religious answer, or  \n3. a brutally honest answer.'

In [6]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [7]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [8]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [9]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'
    course_response = requests.get(course_url)
    course_response.raise_for_status()  # Check for request errors
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [10]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [11]:
from minsearch import Index

index = Index(
    text_fields = ['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [12]:
search_results = index.search(query = 'question', boost_dict={'question': 2.0, 'section': 1.0, 'answer': 1.0}, filter_dict={'course': 'llm-zoomcamp'}, num_results=5)

In [13]:
def search(question, course = 'llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 1.0, 'answer': 1.0}
    filter_dict = {'course': course}
    return index.search(query = question, boost_dict=boost_dict, filter_dict=filter_dict, num_results=5)

In [14]:
search_results = search('question')

In [15]:
search_results

[{'id': '5c4a8d2e60',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'Vector search: should I embed the question, the answer, or both?',
  'answer': 'There\'s no single right answer — it\'s an experiment to run on your dataset. The course shows three options:\n\n- Embed the answer (`text`) only — works because the model captures semantic similarity between questions and their answers.\n- Embed the question only — works because user queries look like the indexed questions.\n- Embed `question + " " + text` — often the best, but produces longer input and slightly more cost.\n\nPick whichever gives the best hit rate / MRR on your ground-truth set. The course materials include a side-by-side comparison.'},
 {'id': '87d4ea4e08',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'How should I choose field weights for minsearch or another search engine?',
  'answer': 'The systematic approach is to evaluate different weight settings again

In [16]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants based on the provided
context.

Use the contetc to find relevant information to provide accurate answers. If the 
answer is not found in the context, say that you don't know.
'''

In [17]:
USER_PROMPT_TEMPALATE = '''
Question: {question}
Context:
{context}
'''

In [18]:
def build_context(search_results):
    lines = []
    for doc in search_results: 
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'] )
        lines.append('')
    return '\n'.join(lines).strip()

In [19]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPALATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [20]:
prompt = build_prompt(question, search_results)

In [21]:
print(prompt)

Question: I just discovered the course. Can I join now?
Context:
Module 2: Vector Search
Q: Vector search: should I embed the question, the answer, or both?
A: There's no single right answer — it's an experiment to run on your dataset. The course shows three options:

- Embed the answer (`text`) only — works because the model captures semantic similarity between questions and their answers.
- Embed the question only — works because user queries look like the indexed questions.
- Embed `question + " " + text` — often the best, but produces longer input and slightly more cost.

Pick whichever gives the best hit rate / MRR on your ground-truth set. The course materials include a side-by-side comparison.

Module 1: RAG
Q: How should I choose field weights for minsearch or another search engine?
A: The systematic approach is to evaluate different weight settings against a ground-truth dataset.

For example:

1. Create a small set of representative questions.
2. Mark which documents should b

In [22]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=prompt,
)

In [23]:
response.output_text

'Yes — you can join now.\n\nThe course materials are available in the repo, so you can start anytime at your own pace. If you get stuck, ask in Slack and follow the course’s question-asking guidelines.\n\nIf you want, I can also help you find the right starting point for the course.'

In [24]:
response.usage

ResponseUsage(input_tokens=565, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=66, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=631)

In [29]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [30]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [26]:
response.output_text

'Yes — you can join now. The course materials are available, and modules are pre-recorded in the course repo, so you can start anytime.\n\nIf you want, I can also help you figure out where to begin.'

In [32]:
def rag( query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(INSTRUCTIONS, prompt, model=model)

In [33]:
answer = rag(question)
print(answer)

Yes — you can still join now.

If you want a certificate, though, you need to submit your project while submissions are still open.
